# sim 엣지 영향력 EDA — S1~S5 단일 노트북

> 계획: `docs/sim_edge_influence_eda_plan.md` · 로직: `src/eval/md/sim_diag.py` · 모델: **v2_sweepA**

network_eda 리포트의 M0 발견(**어텐션 α_r의 87%가 sim_kw/sim_ip에 집중**)을 전용 연구로 확장.

| 섹션 | 질문 | 답 |
|---|---|---|
| S1 | sim_kw 퇴화(일반어 준-완전그래프)? | degree·sweep·공유키워드 |
| S2 | sim_kw vs sim_ip 진짜 변별자? | 이웃성공률 단독 PR-AUC |
| S3 | 전이적 낙관(train↔test 누수)? | sim 교차엣지 ablation |
| S4 | include_sim이 처방 신호 바꾸나? | Δprob direct vs +sim |
| S5 | 깊이(L=3) vs 정규화(IDF)? | A/B/C 재학습 비교 |

## §0. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'; plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_rows', 60)

from src.eval.md.engine import MDEngine, EngineConfig
from src.eval.md import sim_diag as SD

eng = MDEngine(EngineConfig.v2_sweepA()).run_single_inference()
eng.build_mass(); eng.build_ledger('full')
print('모델:', eng.cfg.exp_dir, '| 제품', eng.cache['P'], '키워드', eng.cache['K'])
print('sim 임계: kw>=%d  ip>=%d' % (eng.cache['sim_kw_min_shared'], eng.cache['sim_ip_min_shared']))

## §S1. sim 그래프 구조 진단 — 퇴화 여부

sim_kw = thresholded A²(P-K-P). 공유 키워드≥3이 *너무 낮아* 일반어로 준-완전그래프가 됐는지 본다.

In [ ]:
# S1-1. degree 분포 — sim_kw가 얼마나 밀집했나
deg = SD.sim_degree_stats(eng); display(deg)
h = SD.sim_degree_hist(eng, SD.SIM_KW)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].hist(h, bins=60, color='#16A34A'); ax[0].set_title('sim_kw degree 분포'); ax[0].set_xlabel('이웃 제품 수')
ax[1].hist(SD.sim_degree_hist(eng, SD.SIM_IP), bins=60, color='#F59E0B'); ax[1].set_title('sim_ip degree 분포')
plt.tight_layout(); plt.show()

In [ ]:
# S1-2. min_shared 임계 sweep — 몇으로 올리면 밀집이 풀리나
display(SD.min_shared_sweep(eng))

In [ ]:
# S1-3. sim_kw 엣지를 만든 '공유 키워드' — 일반 허브어가 지배하나
sk, all_generic = SD.sim_kw_shared_keywords(eng, top=20)
display(sk)
print('공유 키워드가 일반 허브어뿐인 엣지 비율(퇴화 지표):', all_generic)

## §S2. 변별력 분해 — sim_kw vs sim_ip 누가 진짜 신호?

이웃 성공률(라벨=train+val만, **test 누수 0**)을 *단일 피처*로 test 성공을 얼마나 가르나.
'어텐션 0.61인데 변별 0'(sim_kw) vs '진짜 변별자'(sim_ip) 가설 검정.

In [ ]:
d2 = SD.discrimination_table(eng)
print('test base rate:', d2.attrs['base_rate_test'])
display(d2)
print('해석: test_PRAUC 가 base보다 크게 높고 4분면(TP/FP/FN/TN) 성공률 격차가 큰 쪽이 진짜 변별자.')

## §S3. 전이적 낙관 ablation — ★누수 직접 검증

train↔test 를 잇는 sim 엣지를 제거하고 재forward → test PR-AUC 하락폭.
**크게 떨어지면** metric이 'train 이웃 조회'에 의존(전이적 낙관) → cold-start 별도평가 필요.
**거의 안 떨어지면** 일반화 신호 → 누수 아님.

In [ ]:
r = SD.ablate_cross_split_sim(eng)
print('full  test PR-AUC :', r['full_test_prauc'])
print('ablated(교차sim 제거):', r['ablated_test_prauc'])
print('하락폭 drop        :', r['drop'])
print('제거된 교차 엣지   :', r['removed'])
tn = SD.train_neighbor_counts(eng, SD.SIM_IP)
print('
test 제품당 train 이웃: 평균 %.1f / train-성공이웃 평균 %.1f' % (
    tn.train_neighbors.mean(), tn.train_succ_neighbors.mean()))
plt.figure(figsize=(5,3)); plt.hist(tn.train_neighbors, bins=40, color='#9b59b6')
plt.title('test 제품별 train sim_ip 이웃 수'); plt.tight_layout(); plt.show()

## §S4. include_sim 처방 토글 — direct vs 배포현실

가상 노드에 sim 즉석 재계산(`include_sim=True`)을 켜면 Δprob(처방 신호)가 어떻게 바뀌나.
killer 후보의 margin·순위가 두 모드에서 얼마나 다른지 → combo/처방 **기본 모드** 결정 근거.

In [ ]:
lg = eng.ledger['full']
killers = [eng.kw_name(k) for k in list(lg.killer)][:15]
cmp = SD.compare_include_sim(eng, killers, n_headroom=20)
display(cmp)
print('rank_flip 합계:', int(cmp['rank_flip'].sum()), '(0이면 순위 불변 → 모드 전환 안전)')
print('평균 증폭 Δ(sim-direct):', round(cmp['Δ(sim-direct)'].mean(), 4))

## §S5. 깊이(L=3) vs 정규화(IDF) — A/B/C 재학습 비교

**재학습은 노트북 밖**(터미널). 여기선 산출 체크포인트를 로드해 한 표로 비교한다.

| 안 | 구성 | 가설 |
|---|---|---|
| A (현행) | sim materialized, L≤2 | 기준선 |
| B | sim 제거 + L=3 재귀 | homophily·오버스무딩↑ → gap↑·변별↓ |
| C | sim + 키워드 IDF/허브 정규화 | 퇴화 해소·변별↑ (★우세 가설) |

학습 명령(예): `python -m src.train.trainer --config <variantC>` → `experiments/results/<dir>/hin_gnn_best.pt`.
체크포인트가 생기면 아래 셀이 각 안을 로드해 PR-AUC·gap·sim_ip 변별력·전이ablation을 비교한다.

### §S5-prep. IDF 정제 미리보기 (재학습 없이)

C안(키워드 IDF 정규화)을 GPU 재학습 *전에* 검증. `S = A·diag(idf)·Aᵀ ≥ τ` 로 sim_kw 재계산.
- **구조**: τ 올릴수록 deg·일반어비율(generic_only)이 어디서 정상화되나 → 재학습 τ 선택
- **변별력**: 정제 sim_kw 이웃성공률 PR-AUC가 현행 0.332에서 회복되나 (sim_ip 0.617이 상한)

> 주의: IDF의 가치는 sim_kw를 스타로 만드는 게 아니라 **노이즈가 어텐션 61% 먹는 걸 막는 것**.
> 변별력은 소폭만 오르고, 진짜 효과(α_r 재분배·gap)는 S5 재학습에서 확인.

In [ ]:
# 구조 sweep — deg·generic_only 가 정상화되는 τ 찾기 (현행: deg 208, generic 0.66)
display(SD.idf_sim_sweep(eng, taus=(2,4,6,8,12,16,24)))
# 변별력 회복 미리보기 — 정제 sim_kw 단독 PR-AUC (현행 0.332 → ? / sim_ip 상한 0.617)
dd = SD.idf_sim_discrimination(eng, taus=(6,8,12,16))
display(dd); print('현행 sim_kw=%.3f  sim_ip 상한=%.3f' % (dd.attrs['현행_sim_kw_PRAUC'], dd.attrs['sim_ip_상한_PRAUC']))
print('→ deg 정상 + generic_only 급락 + 보유율 유지 의 균형점 τ를 재학습 config에 사용 (후보 τ≈12)')

In [ ]:
# A/B/C 비교 (체크포인트 존재 시 자동, 없으면 학습 명령 안내)
VARIANTS = {
    'A_현행(v2_sweepA)':      'experiments/results/v2_sweepA',
    'B_L3재귀(sim제거)':      'experiments/results/v2_L3_nosim',      # 재학습 산출 예정
    'C_simIDF정규화':         'experiments/results/v2_sim_idf',       # 재학습 산출 예정
}
rows = []
for name, d in VARIANTS.items():
    if not os.path.exists(os.path.join(d, 'hin_gnn_best.pt')):
        print('  [미존재]', name, '→', d, '  (학습 필요)')
        continue
    e = MDEngine(EngineConfig(exp_dir=d)).run_single_inference(); e.build_mass()
    dt = SD.discrimination_table(e)
    abl = SD.ablate_cross_split_sim(e) if (SD.SIM_KW in e.cache['eidx'] or SD.SIM_IP in e.cache['eidx']) else {'full_test_prauc':None,'drop':None}
    simip = dt[dt.relation=='sim_ip']['test_PRAUC'].values
    rows.append(dict(variant=name, test_PRAUC=abl['full_test_prauc'],
                     simip_단독PRAUC=(float(simip[0]) if len(simip) else None),
                     전이ablation_drop=abl.get('drop')))
display(pd.DataFrame(rows) if rows else None)
print('
→ 결정 지표: gap 안 키우면서 변별(sim_ip형) 살았나. C가 이기면 정규화 채택.')